# Module 2 — WTI Futures Curve Walkthrough

The goal is not to memorize jargon. The goal is to understand, step by step, what the data is doing and why a producer or trading desk may care.

We will answer one simple question:

> How do WTI futures prices differ across delivery periods?

By the end, you should be able to explain:
- what C1, C2, C3, and C4 mean,
- how to calculate a calendar spread,
- why a positive C1-C4 spread is different from a negative one,
- what backwardation and contango mean,
- why the curve matters to a physical oil producer.

## Step 0 — The mental picture

Think of the futures curve as a row of prices for different delivery positions.

Example:

```
C1   C2   C3   C4
80   79   78   77
```

C1 is the nearest delivery position in the historical EIA series. C2, C3, and C4 are the next positions.

**Important:** C1 is not one permanent contract forever. As time moves forward, the exact contract represented by C1 changes.

## Step 1 — Import the tools we need

We use:
- **pandas** to work with tables,
- the project functions in `src/term_structure.py` to calculate spreads and labels.

The code is shown alongside each step so the calculation and interpretation stay connected.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.append(str(ROOT))

from src.term_structure import (
    add_curve_metrics,
    summarize_curve_regimes,
    select_example_curves,
)

## Step 2 — Load the saved historical curve data

The project includes a saved EIA snapshot so the notebook can be reproduced even if the EIA webpage is temporarily unavailable.

The file contains one row per month and four prices:
- `contract_1`
- `contract_2`
- `contract_3`
- `contract_4`

In [ ]:
curve = pd.read_csv(
    ROOT / "data" / "eia_wti_curve_monthly_2015_2024.csv",
    parse_dates=["date"],
    index_col="date",
)

curve.head()

## Step 3 — Read one row like a human

Before calculating anything, look at one month.

If a row says:

```
C1 = 47.33
C2 = 47.90
C3 = 48.72
C4 = 49.62
```

that means later delivery positions were priced above the nearest delivery position for that observation.

We have not called it backwardation or contango yet. First, we will do the subtraction.

In [ ]:
example = curve.iloc[0]

print("Date:", curve.index[0].date())
print("C1:", example["contract_1"])
print("C2:", example["contract_2"])
print("C3:", example["contract_3"])
print("C4:", example["contract_4"])

## Step 4 — Calculate one C1-C4 spread by hand

The main formula is:

```
C1-C4 spread = C1 price - C4 price
```

Why do this?

Because it gives us one number that tells us whether the near price is above or below the later price.

- Positive result: C1 is above C4.
- Negative result: C1 is below C4.

In [ ]:
c1 = example["contract_1"]
c4 = example["contract_4"]

c1_c4 = c1 - c4

print("C1:", c1)
print("C4:", c4)
print("C1 - C4:", round(c1_c4, 2))

## Step 5 — Let the project calculate all spreads

Now we ask the project function to repeat the same idea for every month.

It creates:
- C1-C2,
- C1-C3,
- C1-C4,
- a simple curve slope,
- a curve label.

In [ ]:
curve_with_metrics = add_curve_metrics(
    curve,
    flat_threshold=0.25,
)

curve_with_metrics.head()

## Step 6 — Understand the labeling rule

This project uses a simple teaching threshold:

```
C1-C4 > +0.25  -> backwardation
C1-C4 < -0.25  -> contango
otherwise       -> flat
```

### Backwardation
Nearby futures are above later futures.

### Contango
Later futures are above nearby futures.

### Flat
The difference is small under this project's threshold.

The **0.25 threshold is not a universal market law**. It is just the cutoff used in this project.

In [ ]:
curve_with_metrics[
    [
        "contract_1",
        "contract_4",
        "c1_c4_spread",
        "curve_regime",
    ]
].head(10)

## Step 7 — Check the logic yourself

Pick any row and ask:

1. Is C1 above C4?
2. If yes, is C1-C4 positive?
3. If the positive difference is above 0.25, does the row say backwardation?

Or:

1. Is C1 below C4?
2. If yes, is C1-C4 negative?
3. If the negative difference is below -0.25, does the row say contango?

Doing this manually for a few rows is the best way to make the terminology stick.

In [ ]:
sample_rows = curve_with_metrics.iloc[[0, 20, 50, 80]]

sample_rows[
    [
        "contract_1",
        "contract_4",
        "c1_c4_spread",
        "curve_regime",
    ]
]

## Step 8 — Count how often each curve shape appeared

Now that each month has a label, we can summarize the historical sample.

This is descriptive analysis.

We are asking:

> How often did each shape appear in this dataset?

We are **not** asking which regime was profitable or which one predicts future prices.

In [ ]:
regime_summary = summarize_curve_regimes(
    curve_with_metrics
)

regime_summary

## Step 9 — Select easy-to-see examples

The code selects:
- the largest positive C1-C4 spread,
- the largest negative C1-C4 spread.

These examples make the shapes visually obvious.

They are selected for teaching, not because they are recommended trades.

In [ ]:
examples = select_example_curves(
    curve_with_metrics
)

examples

## Step 10 — What does this have to do with a producer?

A producer has future oil production arriving over time.

For example:

```
January production
February production
March production
April production
```

A real hedge program needs to think about which futures delivery month best matches each physical production period.

That is why learning the curve matters.

The original hedge model treated futures more simply. This module starts adding the **time dimension** of the futures market.

## Step 11 — Common interpretation mistakes

**Trap 1: "Backwardation means price will rise."**  
No. Backwardation describes prices across delivery months. It is not a guaranteed directional forecast.

**Trap 2: "Contango means price will fall."**  
No. Same reason.

**Trap 3: "C1 is always the same exact futures contract."**  
No. C1 is the nearest delivery position in the historical series, so the exact contract changes through time.

**Trap 4: "The 0.25 threshold is the industry definition."**  
No. It is a simple project setting.

**Trap 5: "One curve shape explains the whole crude market."**  
No. Inventories, production, refinery demand, storage, transport, exports, geopolitics, and other variables can matter too.

## Step 12 — How to explain this in an interview

A simple answer:

> "I added a WTI term-structure module to my producer hedge project. I used the first four historical futures delivery positions, calculated calendar spreads such as C1-C4, and classified observations as backwardation, contango, or flat. The purpose was to move beyond a single futures price and start thinking about contract-month selection for physical production that occurs across time."

If asked why it matters:

> "A producer does not produce all of its oil today, so future production months may need to be matched with different futures contracts. The curve helps describe the price relationships across those delivery periods."

## Step 13 — Mini self-check

You should now be able to answer these without looking back:

1. What does C1 mean?
2. Is C1 a permanent contract?
3. What is the formula for C1-C4?
4. If C1-C4 is positive, which contract is more expensive?
5. What does backwardation mean?
6. What does contango mean?
7. Why is the 0.25 threshold used?
8. Why might a producer care about different delivery months?
9. Does backwardation guarantee higher future WTI prices?
10. What would make this analysis more realistic for a live trading desk?

If any answer feels fuzzy, go back to that step and work through the example again.

## Final takeaway

The entire module can be reduced to this:

> **WTI futures have different prices for different delivery periods. We measure those differences so we can describe the shape of the market across time.**

That is the foundation for later work on:
- contract-month hedge matching,
- roll risk,
- storage economics,
- regime-based hedge analysis,
- fundamentals and trading signals.